# Safety distillation (Qwen2.5-3B-Instruct → Qwen2.5-0.5B-Instruct)


## Data loading

In [ ]:
import os
import tarfile
import pandas as pd

tgz_url = "http://www.linguist.univ-paris-diderot.fr/~mcandito/Ens/safety-kd-data.tgz"
tgz_filename = "safety-kd-data.tgz"
extract_dir = "."

if not os.path.exists(tgz_filename):
    print(f"Downloading {tgz_filename}...")
    !wget -O {tgz_filename} "{tgz_url}"
else:
    print(f"{tgz_filename} already exists.")

try:
    with tarfile.open(tgz_filename, "r:gz") as tar_ref:
        tar_ref.extractall(extract_dir, filter="data")  # safe extraction (no absolute paths / links outside dir)
    print("Extraction complete.")
except Exception as e:
    print(f"Error extracting tar file: {e}")

dfs = {}
base_path = os.path.join(extract_dir, "safety-kd-data")

# Load training and validation files (JSON Lines format)
for filename in ['BTfraction_train', 'BTfraction_val']:
    file_path = os.path.join(base_path, f"{filename}.jsonl")
    if os.path.exists(file_path):
        dfs[filename] = pd.read_json(file_path, lines=True)
        print(f"Loaded {filename}.jsonl ({len(dfs[filename])} rows)")

# Load evaluation files (standard JSON format)
for filename in ['BTeval', 'XSTest']:
    file_path = os.path.join(base_path, "evaluation", f"{filename}.json")
    if os.path.exists(file_path):
        dfs[filename] = pd.read_json(file_path)
        print(f"Loaded {filename} ({len(dfs[filename])} rows)")

print(f"Loaded {len(dfs)} file(s) into the 'dfs' dictionary.")

In [ ]:
from IPython.display import display

print("BTfraction_train:")
display(dfs['BTfraction_train'].head())

print("\nEvaluation : BTeval:")
display(dfs['BTeval'].head())

print("\nEvaluation : XSTest:")
display(dfs['XSTest'].head())

In [ ]:
import torch
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  print("BF16 supported:", torch.cuda.is_bf16_supported())
  print("BF16 natively supported:", torch.cuda.is_bf16_supported(including_emulation=False))
else:
  print("no gpu available")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

STUDENT_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # the Instruct model, not the base model
TEACHER_ID = "Qwen/Qwen2.5-3B-Instruct"

# bf16 only on GPUs with native support (Ampere+, e.g. A100/L4); fp16 on a T4 (emulated bf16 is very slow)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("using dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(STUDENT_ID)
model = AutoModelForCausalLM.from_pretrained(
    STUDENT_ID,
    dtype=dtype,          # `torch_dtype` is deprecated in recent transformers
    device_map="auto",
)

In [ ]:
# checking the counts of the weights for each data type
from collections import Counter

dtype_counts = Counter(
    str(param.dtype)
    for param in model.parameters()
)

print(dtype_counts)